# Fine-tuning de XLM-R estabilizado (complemento del OE4) — cuaderno de XLM-R, versión 3

**Para qué sirve.** En la segunda tanda, XLM-R fue el único modelo con **entrenamientos que colapsaron**: el modelo final terminó prediciendo una sola intención (F1 de la partición 0.04 a 0.13) aunque su validación interna fue normal. Los datos muestran que no es un error del corpus ni de una configuración: la misma receta funciona en otras variantes y los colapsos aparecen en 4 de 75 entrenamientos finales (5%) y en 11 de 150 de validación (7%) de XLM-R, y mucho más con la tasa 5e-5 (7 de 15 con 10 épocas). Es la inestabilidad conocida del ajuste fino de estos modelos con pocos datos (Mosbach et al., 2021).

Este cuaderno repite el XLM-R de la segunda tanda con un **procedimiento estabilizado**, declarado antes de ver los resultados:

| Medida | Antes (2.ª tanda) | Ahora |
|---|---|---|
| Tasas de aprendizaje | 3e-5 y 5e-5 (más 1e-5, 2e-5 de la 1.ª tanda) | **2e-5 y 3e-5** (5e-5 colapsó con frecuencia) |
| Épocas | 10, 15 y 20 (eligió 20 en 13 de 15: borde) | **15, 20 y 25** |
| Calentamiento (warmup) | 10% de los pasos | **20%** |
| Detección de colapso | ninguna | **Sí**: si la pérdida de entrenamiento sigue en el nivel del azar (≥ 1.85, 0.95 × ln 7) al 30% de las épocas, se **descarta ese entrenamiento y se repite con otra semilla** (hasta 2 reinicios). Usa solo la pérdida de entrenamiento: no mira etiquetas de prueba ni validación. |
| Registro | solo predicciones | **pérdida por época de cada entrenamiento**, para poder confirmar la causa |
| Puntajes de validación previos | se reutilizaban | **no se reutilizan**: todas las combinaciones se recalculan con el procedimiento nuevo |

Las tres variantes son las mismas de la segunda tanda (rejilla propia, + Generate-then-Refine y pooling del OE3), con la misma regla: `gtr` y `pooling` usan los hiperparámetros que eligió la primera, de modo que cada una cambia una sola cosa.

**Reglas fijadas antes de ver resultados.**
- Los hiperparámetros se eligen solo con validación interna del entrenamiento, nunca con la prueba.
- La regla de colapso y sus umbrales (`UMBRAL_COLAPSO`, `MAX_REINICIOS`) se aplican **a todos los entrenamientos por igual** (validación y finales). Si tras los reinicios sigue colapsado, se acepta ese resultado y se informa.
- **Se reporta junto a la segunda tanda, no en su lugar:** la variante estabilizada es una receta nueva y ambas quedan en el informe (la anterior no se borra ni se corrige).
- Se informa cuántos reinicios hubo y en qué entrenamientos.
- **No usa la API de Claude (no gasta dinero).**

**Qué NO garantiza.** Estabilizar no equivale a mejorar el F1: puede que XLM-R quede igual o por debajo de la segunda tanda. Tampoco garantiza que no haya colapsos; garantiza que, si ocurren, se detectan, se reintentan y se registran.

**Tiempo estimado en T4 (no medido):** unas 5 a 6 horas (la variante propia ~3.5 h; `gtr` ~1 h; `pooling` ~0.5 h), algo más si hay reinicios. El avance se guarda en tu Drive: si Colab se desconecta, vuelve a correr desde la celda de configuración y retoma.

**Uso:** Entorno de ejecución → GPU (T4). Corre las celdas en orden. Para una comprobación rápida, `PRUEBA = True` en la configuración.

## 0. Montar Google Drive (aquí queda el avance y los resultados)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. Clonar tu repositorio

Si falta algún archivo, haz `git push` de lo nuevo y vuelve a correr esta celda.

In [ ]:
import os
GITHUB_USUARIO = "LhiaHC"

%cd /content
!rm -rf /content/shiwilu-tesis
!git clone https://github.com/{GITHUB_USUARIO}/shiwilu-tesis.git
%cd /content/shiwilu-tesis
!git log --oneline -3

REPO = '/content/shiwilu-tesis'
requeridos = [
    f'{REPO}/corpus/corpus_shiwilu_final.csv',
    f'{REPO}/oe2_aumento_de_datos/particiones/folds_fijos.csv',
    f'{REPO}/oe2_aumento_de_datos/evaluacion/resultados/cv_interna/validacion_cruzada_predicciones_sin_puntuacion_cv_interna_c120.csv',
    f'{REPO}/oe2_aumento_de_datos/evaluacion/resultados/cv_interna/volumen_sintetico_sin_puntuacion_cv_interna_generate_then_refine_c120.csv',
    f'{REPO}/oe2_aumento_de_datos/tecnicas_aumento/salidas/en_linea_marcadores_fuentes/generate_then_refine/fold0_pool.csv',
    f'{REPO}/oe4_sintesis/resultados/ngramas_predicciones.csv',
    f'{REPO}/oe4_sintesis/finetuning/resultados/predicciones_xlmr_ampliada.csv',
    f'{REPO}/shiwilu/clasificacion.py',
]
faltan = [r for r in requeridos if not os.path.exists(r)]
assert not faltan, 'Faltan archivos en el clon: ' + ', '.join(faltan)
print('OK')

## 2. Instalar dependencias y comprobar la GPU

`torchao` se desinstala a propósito: Colab trae una versión vieja que choca con `transformers` (error «Found an incompatible version of torchao»). Si ese error reaparece, corre solo esta celda otra vez y luego **Entorno de ejecución → Reiniciar sesión** (no pierdes nada: el avance está en Drive).

In [ ]:
!pip install -q transformers accelerate scikit-learn sentencepiece
!pip uninstall -y -q torchao
import torch
print('GPU disponible:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'SIN GPU (cambia el tipo de entorno de ejecucion)')

## 3. Configuración (corre esta celda en cada sesión)

- `MODELO`: solo nombra los archivos de este cuaderno (`xlmr_estable`); el modelo es XLM-R (`xlm-roberta-base`).
- `LRS` y `EPOCAS`: la rejilla de esta versión (tasas 2e-5 y 3e-5; 15, 20 y 25 épocas). Se quitó 5e-5, que colapsó con frecuencia en la validación interna, y se agregó 25 épocas porque la rejilla anterior eligió el máximo.
- `WARMUP`: fracción de los pasos en que la tasa de aprendizaje sube desde 0 (20%, antes 10%).
- `UMBRAL_COLAPSO` y `MAX_REINICIOS`: regla de detección de colapso. Si la pérdida de entrenamiento sigue en el nivel del azar (≥ 1.85) al 30% de las épocas, ese entrenamiento se descarta y se repite con otra semilla, hasta 2 veces; el último intento se acepta y se informa.
- `PRUEBA = True`: corrida mínima para comprobar que todo funciona (1 semilla, 1 partición, 1 combinación, 4 épocas).

In [ ]:
import os, re, sys, math, time, random, glob, shutil
import numpy as np, pandas as pd, torch
from types import SimpleNamespace

MODELO = "xlmr_estable"                      # nombre de los archivos de este cuaderno (el modelo es XLM-R)
HF_ID = "xlm-roberta-base"
PREVIO = "xlmr_rejilla2_completo"            # fine-tuning anterior de XLM-R (primera tanda) para comparar
POOLING = "combinacion_capas"                # el que mejor funciono congelado en el OE3
LRS = [2e-5, 3e-5]                           # sin 5e-5: colapso con frecuencia en la validacion interna
EPOCAS = [15, 20, 25]                        # incluye 25 porque la rejilla anterior eligio el maximo (20) en 13 de 15
CONFIGS_NUEVAS = [(lr, ep) for lr in LRS for ep in EPOCAS]
WARMUP = 0.20                                # fraccion de los pasos de calentamiento (antes 0.10)
UMBRAL_COLAPSO = 1.85                        # 0.95 * ln(7) = nivel de pérdida de predecir siempre lo mismo
MAX_REINICIOS = 2                            # reintentos con otra semilla si se detecta colapso
VARIANTES = ["ampliada", "gtr", "pooling"]
SEMILLAS = [0, 1, 2]
FOLDS = [0, 1, 2, 3, 4]
N_INTERNOS = 2
BATCH = 16
MAX_LEN = 64
REPETICIONES_BOOTSTRAP = 2000
REUTILIZAR_PUNTAJES_PREVIOS = False          # todas las combinaciones se recalculan con el procedimiento estabilizado
PRUEBA = False                               # True = comprobacion rapida

if PRUEBA:
    SEMILLAS, FOLDS, N_INTERNOS, CONFIGS_NUEVAS = [0], [0], 1, [(3e-5, 4)]

SUFIJO = "_prueba" if PRUEBA else ""
DIR_DRIVE = f"/content/drive/MyDrive/shiwilu_finetuning_v3_{MODELO}" + SUFIJO     # avance y resultados (sobrevive a desconexiones)
DIR_RUNS = DIR_DRIVE + "/corridas"
os.makedirs(DIR_RUNS, exist_ok=True)
DEV = "cuda" if torch.cuda.is_available() else "cpu"
USAR_FP16 = DEV == "cuda"
print("Dispositivo:", DEV, "| modelo:", MODELO, "| variantes:", VARIANTES, "| semillas:", SEMILLAS, "| particiones:", FOLDS)
print("Combinaciones a probar:", CONFIGS_NUEVAS, "| pooling de la variante 3:", POOLING)
print(f"Calentamiento {WARMUP:.0%} | colapso si la perdida >= {UMBRAL_COLAPSO} al 30% de las epocas | hasta {MAX_REINICIOS} reinicios")

## 4. Cargar los datos (los mismos del OE2) y las oraciones sintéticas de Generate-then-Refine

- Corpus, texto normalizado y particiones congeladas: idénticos a los 12 experimentos. `pos` es la posición de cada oración en el corpus (0 a 699).
- **GtR de cada partición:** se lee la caché del OE2 (`fold<N>_pool*.csv`, 6 lotes de 20 por intención, sin llamar a la API) y se arma igual que en el OE2: se quitan repetidas y copias de oraciones reales del entrenamiento, se conservan las aprobadas por los 3 filtros y se descartan las que copian una oración de la prueba. El cuaderno **comprueba que el volumen coincide con el del OE2** (493, 475, 510, 508 y 503 oraciones por partición).

In [ ]:
sys.path.insert(0, REPO)
from shiwilu.clasificacion import cargar_corpus, cargar_corpus_normalizado, cargar_folds, _normalizar_estricto, quitar_puntuacion
from shiwilu.taxonomia import INTENCIONES

CLASES = sorted(INTENCIONES)                                  # AFI, DES, EMO, NEG, PRG, REQUEST, SAL
K = len(CLASES)
corpus = cargar_corpus()
assert corpus.index.equals(pd.RangeIndex(len(corpus)))
texto = cargar_corpus_normalizado()["shiwilu"].astype(str).to_numpy()
y = corpus["intencion"].to_numpy()
Y_IDX = np.array([CLASES.index(c) for c in y])
folds = cargar_folds(corpus)
grupos = np.array([_normalizar_estricto(t) for t in corpus["shiwilu"]])
print(len(corpus), "oraciones |", K, "intenciones | tamanos de prueba por particion:", [int((folds == f).sum()) for f in range(5)])

def f1_por_clase(yv, pv):
    cm = np.bincount(yv * K + pv, minlength=K * K).reshape(K, K)
    tp = np.diag(cm); fp = cm.sum(0) - tp; fn = cm.sum(1) - tp
    den = 2 * tp + fp + fn
    return np.where(den > 0, 2 * tp / np.maximum(den, 1), 0.0)

def f1_macro_idx(yv, pv):
    return float(f1_por_clase(yv, pv).mean())

# ---- Generate-then-Refine de cada particion (cache del OE2, sin API)
RUTA_GTR = f"{REPO}/oe2_aumento_de_datos/tecnicas_aumento/salidas/en_linea_marcadores_fuentes/generate_then_refine"
def gtr_de_particion(f):
    claves_pool = set(grupos[folds != f]); claves_test = set(grupos[folds == f])
    g = pd.concat([pd.read_csv(f"{RUTA_GTR}/fold{f}_pool{'' if j == 0 else f'_l{j}'}.csv") for j in range(6)], ignore_index=True)
    claves = g["shiwilu"].map(_normalizar_estricto)
    repetida = pd.Series(list(zip(g["intencion"], claves))).duplicated().to_numpy()
    copia = claves.isin(claves_pool).to_numpy()
    g = g[~(repetida | copia)].reset_index(drop=True)
    ap = g[g["estado_filtro"] == "aprobado"].reset_index(drop=True)
    return ap[~ap["shiwilu"].map(_normalizar_estricto).isin(claves_test)].reset_index(drop=True)

TEXTOS = list(texto); Y_ALL = list(Y_IDX); EXTRA = {}
for f in range(5):
    s = gtr_de_particion(f); ini = len(TEXTOS)
    TEXTOS += [quitar_puntuacion(t, minusculas=True, quitar_tildes=True) for t in s["shiwilu"]]
    Y_ALL += [CLASES.index(c) for c in s["intencion"]]
    EXTRA[f] = np.arange(ini, len(TEXTOS))
Y_ALL = np.array(Y_ALL)
vol = pd.read_csv(f"{REPO}/oe2_aumento_de_datos/evaluacion/resultados/cv_interna/volumen_sintetico_sin_puntuacion_cv_interna_generate_then_refine_c120.csv")
mios = [len(EXTRA[f]) for f in range(5)]
print("Sinteticos GtR por particion (aqui):", mios, "| OE2:", vol["sinteticos"].tolist())
assert mios == vol["sinteticos"].tolist(), "el volumen de GtR NO coincide con el del OE2: no seguir"
print("OK: el aumento coincide con el del OE2")
print("Ejemplos:", [TEXTOS[i] for i in EXTRA[0][:3]])

# ---- puntajes de validacion de la corrida anterior (se reutilizan; no se vuelven a entrenar)
sel_prev = pd.read_csv(f"{REPO}/oe4_sintesis/finetuning/resultados/seleccion_hiperparametros_{PREVIO}.csv")
ex = sel_prev["archivo"].str.extract(r"_s(\d+)_f(\d+)\.csv")
sel_prev["semilla"], sel_prev["fold"] = ex[0].astype(int), ex[1].astype(int)
print("Combinaciones ya probadas en la corrida anterior:", sorted(set(zip(sel_prev.lr, sel_prev.epocas))))

## 5. Funciones de entrenamiento

- `ampliada` y `gtr` usan la cabeza de clasificación estándar de Hugging Face (vector [CLS]), igual que la corrida anterior.
- `pooling` usa el modelo base más una capa lineal sobre el pooling elegido (calculado con la misma lógica del OE3: mean, max o promedio de las 4 últimas capas).
- En `gtr`, el entrenamiento mezcla las oraciones reales de la partición con sus sintéticas; la evaluación es siempre sobre las oraciones reales de prueba.

In [ ]:
import json
from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.model_selection import StratifiedGroupKFold

_tok = AutoTokenizer.from_pretrained(HF_ID)
_enc = _tok(list(TEXTOS), padding=True, truncation=True, max_length=MAX_LEN, return_tensors="pt")
IDS, MASK = _enc["input_ids"], _enc["attention_mask"]
print(f"ancho de secuencia (piezas): {IDS.shape[1]} | oraciones tokenizadas (reales + sinteticas): {IDS.shape[0]}")

def fijar_semilla(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

def agrupar(salida, mascara, estrategia):
    m = mascara.unsqueeze(-1).float()
    if estrategia == "cls":
        return salida.last_hidden_state[:, 0, :].float()
    if estrategia == "mean_pooling":
        h = salida.last_hidden_state.float()
        return (h * m).sum(1) / m.sum(1).clamp(min=1e-9)
    if estrategia == "max_pooling":
        h = salida.last_hidden_state.float()
        return h.masked_fill(m == 0, float("-inf")).max(dim=1).values
    if estrategia == "combinacion_capas":      # mean pooling sobre el promedio de las ultimas 4 capas (como en el OE3)
        h = torch.stack(salida.hidden_states[-4:], dim=0).float().mean(0)
        return (h * m).sum(1) / m.sum(1).clamp(min=1e-9)
    raise ValueError(estrategia)

class RedPooling(torch.nn.Module):
    def __init__(self, hf_id, estrategia, n_clases):
        super().__init__()
        self.estrategia = estrategia
        self.enc = AutoModel.from_pretrained(hf_id, output_hidden_states=(estrategia == "combinacion_capas"))
        self.drop = torch.nn.Dropout(0.1)
        self.cabeza = torch.nn.Linear(self.enc.config.hidden_size, n_clases)
    def forward(self, input_ids, attention_mask, labels=None):
        v = agrupar(self.enc(input_ids=input_ids, attention_mask=attention_mask), attention_mask, self.estrategia)
        logits = self.cabeza(self.drop(v))
        loss = torch.nn.functional.cross_entropy(logits.float(), labels) if labels is not None else None
        return SimpleNamespace(loss=loss, logits=logits)

def crear_red(variante):
    if variante == "pooling":
        return RedPooling(HF_ID, POOLING, K)
    return AutoModelForSequenceClassification.from_pretrained(HF_ID, num_labels=K, id2label=dict(enumerate(CLASES)), label2id={c: i for i, c in enumerate(CLASES)})

LOG = []          # un registro por entrenamiento: reinicios y perdida por epoca

def entrenar_una_vez(idx_tr, idx_evals, lr, epocas, semilla, variante, detectar):
    # entrena una vez. Devuelve (predicciones o None si se detecto colapso, perdida media por epoca)
    fijar_semilla(semilla)
    net = crear_red(variante).to(DEV)
    params = [p for p in net.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=lr, weight_decay=0.01)
    pasos = epocas * math.ceil(len(idx_tr) / BATCH)
    sch = get_linear_schedule_with_warmup(opt, int(WARMUP * pasos), pasos)
    scaler = torch.amp.GradScaler("cuda", enabled=USAR_FP16)
    g = torch.Generator().manual_seed(semilla)
    ytr = torch.as_tensor(Y_ALL[idx_tr])
    chk = max(3, math.ceil(0.3 * epocas))          # epoca en la que se revisa si el entrenamiento esta aprendiendo
    perdidas, colapso = [], False
    net.train()
    for e in range(epocas):
        perm = torch.randperm(len(idx_tr), generator=g)
        acum, nb = 0.0, 0
        for k in range(0, len(perm), BATCH):
            b = perm[k:k + BATCH]
            ii = torch.as_tensor(idx_tr[b.numpy()])
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USAR_FP16):
                salida = net(input_ids=IDS[ii].to(DEV), attention_mask=MASK[ii].to(DEV), labels=ytr[b].to(DEV))
            if not torch.isfinite(salida.loss):
                colapso = True; break                   # la perdida se volvio NaN o infinita
            opt.zero_grad(set_to_none=True)
            scaler.scale(salida.loss).backward()
            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(params, 1.0)
            scaler.step(opt); scaler.update(); sch.step()
            acum += float(salida.loss.detach()); nb += 1
        if colapso:
            break
        perdidas.append(acum / max(nb, 1))
        if detectar and e + 1 == chk and perdidas[-1] >= UMBRAL_COLAPSO:
            colapso = True; break                       # sigue en el nivel del azar: el modelo no esta aprendiendo
    if colapso and detectar:
        del net, opt
        if DEV == "cuda":
            torch.cuda.empty_cache()
        return None, perdidas
    net.eval(); salidas = []
    with torch.no_grad():
        for idx in idx_evals:
            pr = []
            for k in range(0, len(idx), 64):
                ii = torch.as_tensor(idx[k:k + 64])
                with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USAR_FP16):
                    lg = net(input_ids=IDS[ii].to(DEV), attention_mask=MASK[ii].to(DEV)).logits
                pr.append(lg.argmax(-1).cpu().numpy())
            salidas.append(np.concatenate(pr))
    del net, opt
    if DEV == "cuda":
        torch.cuda.empty_cache()
    return salidas, perdidas

def entrenar_y_predecir(idx_tr, idx_evals, lr, epocas, semilla, variante):
    # entrena; si se detecta colapso, repite con otra semilla (hasta MAX_REINICIOS). El ultimo intento no se descarta: se informa lo que salga.
    reinicios, perdidas_descartadas = 0, []
    for intento in range(MAX_REINICIOS + 1):
        ultimo = intento == MAX_REINICIOS
        salidas, perdidas = entrenar_una_vez(idx_tr, idx_evals, lr, epocas, semilla + 1000 * intento, variante, detectar=not ultimo)
        if salidas is not None:
            break
        reinicios += 1
        perdidas_descartadas.append([round(x, 3) for x in perdidas])
    LOG.append({"lr": lr, "epocas": epocas, "n_entrenamiento": len(idx_tr), "reinicios": reinicios,
                "perdida_en_revision": round(perdidas[max(3, math.ceil(0.3 * epocas)) - 1], 4) if len(perdidas) >= max(3, math.ceil(0.3 * epocas)) else None,
                "perdida_final": round(perdidas[-1], 4) if perdidas else None,
                "perdidas_por_epoca": ";".join(f"{x:.3f}" for x in perdidas), "intentos_descartados": json.dumps(perdidas_descartadas)})
    return salidas

def salida_csv(variante, semilla, f, idx_test, pred, lr, ep, f1_val, n_sint):
    return pd.DataFrame({"modelo": MODELO, "variante": variante, "semilla": semilla, "fold": f, "pos": idx_test, "real": y[idx_test],
                         "prediccion": [CLASES[i] for i in pred], "lr": lr, "epocas": ep, "f1_validacion_interna": f1_val, "n_sinteticos": n_sint})

def correr_ampliada(semilla, f):
    # elige (lr, epocas) con validacion interna del entrenamiento (reutilizando los puntajes ya calculados de las 6 combinaciones previas)
    idx_test = np.where(folds == f)[0]
    idx_pool = np.where(folds != f)[0]
    cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=semilla)
    internos = list(cv.split(idx_pool, Y_IDX[idx_pool], groups=grupos[idx_pool]))[:N_INTERNOS]
    candidatos = []
    if REUTILIZAR_PUNTAJES_PREVIOS:
        prev = sel_prev[(sel_prev.semilla == semilla) & (sel_prev.fold == f)]
        assert len(prev) >= 6, f"faltan puntajes previos de la semilla {semilla} particion {f}"
        candidatos += [{"lr": r.lr, "epocas": int(r.epocas), "f1_validacion_interna": r.f1_validacion_interna, "origen": "previo"} for r in prev.itertuples()]
    for lr, ep in CONFIGS_NUEVAS:
        f1s = []
        for tr, va in internos:
            p = entrenar_y_predecir(idx_pool[tr], [idx_pool[va]], lr, ep, semilla, "ampliada")[0]
            f1s.append(f1_macro_idx(Y_IDX[idx_pool[va]], p))
        candidatos.append({"lr": lr, "epocas": ep, "f1_validacion_interna": float(np.mean(f1s)), "origen": "nuevo"})
    cand = pd.DataFrame(candidatos)
    mejor = max(candidatos, key=lambda c: (c["f1_validacion_interna"], -c["epocas"], -c["lr"]))   # desempate: menos epocas, menor lr
    pred = entrenar_y_predecir(idx_pool, [idx_test], mejor["lr"], mejor["epocas"], semilla, "ampliada")[0]
    return salida_csv("ampliada", semilla, f, idx_test, pred, mejor["lr"], mejor["epocas"], mejor["f1_validacion_interna"], 0), cand

def config_de_ampliada(semilla, f):
    ruta = f"{DIR_RUNS}/{MODELO}_ampliada_s{semilla}_f{f}.csv"
    assert os.path.exists(ruta), f"falta la variante 'ampliada' de la semilla {semilla} particion {f}: corre primero esa variante"
    d = pd.read_csv(ruta).iloc[0]
    return float(d.lr), int(d.epocas), float(d.f1_validacion_interna)

def correr_con_receta_de_ampliada(variante, semilla, f):
    # gtr y pooling: misma receta (lr, epocas) que eligio la variante 'ampliada' para esta semilla y particion
    lr, ep, f1v = config_de_ampliada(semilla, f)
    idx_test = np.where(folds == f)[0]
    idx_pool = np.where(folds != f)[0]
    idx_tr = np.concatenate([idx_pool, EXTRA[f]]) if variante == "gtr" else idx_pool
    pred = entrenar_y_predecir(idx_tr, [idx_test], lr, ep, semilla, variante)[0]
    return salida_csv(variante, semilla, f, idx_test, pred, lr, ep, f1v, len(EXTRA[f]) if variante == "gtr" else 0), None

## 6. Correr los experimentos (retoma sola si Colab se desconecta)

Orden: primero la variante `ampliada` (las otras dependen de ella), luego `gtr` y `pooling`. Para cada variante, semilla y partición se guarda un archivo en tu Drive en cuanto termina; si vuelves a correr la celda, salta lo que ya existe. Cada partición imprime la combinación elegida, su F1 de validación interna (no es el F1 de prueba) y el tiempo restante estimado.

In [ ]:
t_ini = time.time()
for variante in VARIANTES:
    pendientes = [(s, f) for s in SEMILLAS for f in FOLDS if not os.path.exists(f"{DIR_RUNS}/{MODELO}_{variante}_s{s}_f{f}.csv")]
    print(f"\n=== variante '{variante}': {len(pendientes)} particiones pendientes de {len(SEMILLAS) * len(FOLDS)} ===")
    duraciones = []
    for semilla, f in pendientes:
        t0 = time.time(); n_log = len(LOG)
        if variante == "ampliada":
            salida, cand = correr_ampliada(semilla, f)
            cand.to_csv(f"{DIR_RUNS}/seleccion_{MODELO}_ampliada_s{semilla}_f{f}.csv", index=False)
        else:
            salida, _ = correr_con_receta_de_ampliada(variante, semilla, f)
        salida.to_csv(f"{DIR_RUNS}/{MODELO}_{variante}_s{semilla}_f{f}.csv", index=False)
        reg = pd.DataFrame(LOG[n_log:]).assign(variante=variante, semilla=semilla, fold=f)
        reg.to_csv(f"{DIR_RUNS}/log_{MODELO}_{variante}_s{semilla}_f{f}.csv", index=False)
        n_reinicios = int(reg.reinicios.sum())
        f1p = f1_macro_idx(Y_IDX[salida.pos.to_numpy()], np.array([CLASES.index(c) for c in salida.prediccion]))
        duraciones.append(time.time() - t0)
        resta = (len(pendientes) - len(duraciones)) * np.mean(duraciones) / 60
        print(f"{MODELO} {variante} semilla {semilla} particion {f}: lr={salida.lr.iloc[0]:g}, epocas={salida.epocas.iloc[0]}, "
              f"F1 val. interna={salida.f1_validacion_interna.iloc[0]:.3f} | F1 prueba de esta particion={f1p:.3f} | reinicios: {n_reinicios} | {duraciones[-1]:.0f} s | restan ~{resta:.0f} min de esta variante")
print(f"\nTiempo total de esta sesion: {(time.time() - t_ini) / 60:.1f} min")

## 7. Resultados y comparaciones pareadas

Para cada variante se unen las 5 particiones (700 predicciones) y se calcula el **F1 macro agrupado** (promedio de las semillas), con IC 95% bootstrap. Comparaciones, todas **pareadas** (se remuestrean las mismas oraciones):

1. contra el **fine-tuning anterior** de este modelo (la referencia de la que se parte);
2. contra el **baseline congelado** del OE2;
3. contra el **clasificador de n-gramas** (0.751);
4. las variantes `gtr` y `pooling` contra la variante `ampliada` (misma receta, un solo cambio).

In [ ]:
base_todo = pd.read_csv(f"{REPO}/oe2_aumento_de_datos/evaluacion/resultados/cv_interna/validacion_cruzada_predicciones_sin_puntuacion_cv_interna_c120.csv")
b = base_todo[(base_todo.modelo == "xlmr") & (base_todo.tecnica == "sin_aumento")].sort_values("pos")
assert len(b) == 700 and (b.real.to_numpy() == y).all()
P_BASE = np.array([CLASES.index(c) for c in b.prediccion])
ng = pd.read_csv(f"{REPO}/oe4_sintesis/resultados/ngramas_predicciones.csv").sort_values("pos")
P_NG = np.array([CLASES.index(c) for c in ng.prediccion])
prev = pd.read_csv(f"{REPO}/oe4_sintesis/finetuning/resultados/predicciones_{PREVIO}.csv").sort_values(["semilla", "pos"])
P_PREV = np.stack([np.array([CLASES.index(c) for c in g.prediccion]) for _, g in prev.groupby("semilla")])

def cargar(variante):
    preds = {}
    for semilla in SEMILLAS:
        partes = [pd.read_csv(r) for f in FOLDS for r in [f"{DIR_RUNS}/{MODELO}_{variante}_s{semilla}_f{f}.csv"] if os.path.exists(r)]
        if partes:
            d = pd.concat(partes).sort_values("pos")
            if len(d) == 700 or PRUEBA:
                preds[semilla] = (d.pos.to_numpy(), np.array([CLASES.index(c) for c in d.prediccion]))
    return preds

DISP = {v: cargar(v) for v in VARIANTES}
DISP = {v: p for v, p in DISP.items() if p}
assert DISP, "Aun no hay ninguna variante con las 700 oraciones: termina la celda 6 (o vuelve a correrla) antes de pedir resultados"
print("Variantes con resultados completos:", {v: f"{len(p)} semillas" for v, p in DISP.items()})
rng = np.random.default_rng(42)

def pareada(yv, A, B_):
    # A y B_: (semillas x n). Devuelve (dif, bajo, alto, distinguible) del F1 macro, promediando semillas
    A, B_ = np.atleast_2d(A), np.atleast_2d(B_); n = len(yv); d = []
    for _ in range(REPETICIONES_BOOTSTRAP):
        i = rng.integers(0, n, n)
        d.append(np.mean([f1_macro_idx(yv[i], p[i]) for p in A]) - np.mean([f1_macro_idx(yv[i], p[i]) for p in B_]))
    dif = np.mean([f1_macro_idx(yv, p) for p in A]) - np.mean([f1_macro_idx(yv, p) for p in B_])
    lo, hi = np.percentile(d, [2.5, 97.5])
    return float(dif), float(lo), float(hi), bool(lo > 0 or hi < 0)

filas, por_clase = [], {}
P_V2 = {}
for _v in ("ampliada", "gtr", "pooling"):
    _d = pd.read_csv(f"{REPO}/oe4_sintesis/finetuning/resultados/predicciones_xlmr_{_v}.csv").sort_values(["semilla", "pos"])
    P_V2[_v] = np.stack([np.array([CLASES.index(c) for c in g.prediccion]) for _, g in _d.groupby("semilla")])
refs = {"f1_finetuning_anterior": P_PREV, "f1_baseline_congelado": P_BASE[None, :], "f1_ngramas": P_NG[None, :],
        "f1_v2_ampliada": P_V2["ampliada"], "f1_v2_gtr": P_V2["gtr"], "f1_v2_pooling": P_V2["pooling"]}
for v, preds in DISP.items():
    pos = next(iter(preds.values()))[0]; yv = Y_IDX[pos]
    P = np.stack([p for (_, p) in preds.values()]); f1s = [f1_macro_idx(yv, p) for p in P]
    n = len(yv); medias = []
    for _ in range(REPETICIONES_BOOTSTRAP):
        i = rng.integers(0, n, n); medias.append(np.mean([f1_macro_idx(yv[i], p[i]) for p in P]))
    fila = {"modelo": MODELO, "variante": v, "semillas": len(P), "f1_macro": float(np.mean(f1s)), "ic95_bajo": np.percentile(medias, 2.5), "ic95_alto": np.percentile(medias, 97.5),
            "sd_entre_semillas": float(np.std(f1s)), "minimo": min(f1s), "maximo": max(f1s)}
    for nombre, R in refs.items():
        dif, lo, hi, dist = pareada(yv, P, R[:, pos])
        fila.update({f"{nombre}": float(np.mean([f1_macro_idx(yv, p) for p in R[:, pos]])), f"dif_vs_{nombre[3:]}": dif, f"dif_vs_{nombre[3:]}_ic95_bajo": lo, f"dif_vs_{nombre[3:]}_ic95_alto": hi, f"distinguible_vs_{nombre[3:]}": dist})
    if v != "ampliada" and "ampliada" in DISP:
        pa = np.stack([p for (_, p) in DISP["ampliada"].values()])
        dif, lo, hi, dist = pareada(yv, P, pa)
        fila.update({"dif_vs_ampliada": dif, "dif_vs_ampliada_ic95_bajo": lo, "dif_vs_ampliada_ic95_alto": hi, "distinguible_vs_ampliada": dist})
    filas.append(fila)
    por_clase[f"{MODELO} {v}"] = np.mean([f1_por_clase(yv, p) for p in P], axis=0)
por_clase[f"{MODELO} ft anterior"] = np.mean([f1_por_clase(Y_IDX, p) for p in P_PREV], axis=0)
RES = pd.DataFrame(filas)
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 40)
cols = ["variante", "semillas", "f1_macro", "ic95_bajo", "ic95_alto", "sd_entre_semillas", "f1_finetuning_anterior", "dif_vs_finetuning_anterior", "dif_vs_finetuning_anterior_ic95_bajo", "dif_vs_finetuning_anterior_ic95_alto", "distinguible_vs_finetuning_anterior"]
print("\n=== F1 macro de cada variante y diferencia pareada frente al fine-tuning anterior ===")
print(RES[[c for c in cols if c in RES]].round(3).to_string(index=False))
print("\n=== Frente al baseline congelado y a los n-gramas ===")
print(RES[["variante", "f1_baseline_congelado", "dif_vs_baseline_congelado", "dif_vs_baseline_congelado_ic95_bajo", "dif_vs_baseline_congelado_ic95_alto", "dif_vs_ngramas", "dif_vs_ngramas_ic95_bajo", "dif_vs_ngramas_ic95_alto"]].round(3).to_string(index=False))
if "dif_vs_ampliada" in RES:
    print("\n=== gtr y pooling frente a la variante 'ampliada' (misma receta, un solo cambio) ===")
    print(RES[RES.variante != "ampliada"][["variante", "dif_vs_ampliada", "dif_vs_ampliada_ic95_bajo", "dif_vs_ampliada_ic95_alto", "distinguible_vs_ampliada"]].round(3).to_string(index=False))
print("\nF1 por intencion (promedio de semillas):")
print(pd.DataFrame(por_clase, index=CLASES).round(3).to_string())

# aviso de borde de la rejilla ampliada
if "ampliada" in DISP:
    el = pd.concat([pd.read_csv(r) for r in sorted(glob.glob(f"{DIR_RUNS}/{MODELO}_ampliada_s*_f*.csv"))]).drop_duplicates(["semilla", "fold"])
    print("\nCombinaciones elegidas por la variante 'ampliada' (cuantas de", len(el), "particiones):")
    print(el.groupby(["lr", "epocas"]).size().to_string())
    print(f"Eligio el maximo de epocas probado ({max(e for _, e in CONFIGS_NUEVAS)}) en {(el.epocas == max(e for _, e in CONFIGS_NUEVAS)).sum()} de {len(el)}: si es casi siempre, el resultado sigue siendo conservador.")

# --- reinicios y particiones colapsadas
logs = [pd.read_csv(r) for r in sorted(glob.glob(f"{DIR_RUNS}/log_{MODELO}_*_s*_f*.csv"))]
if logs:
    LG = pd.concat(logs)
    print("\n=== Detección de colapso: entrenamientos con reinicios ===")
    print(f"Entrenamientos registrados: {len(LG)} | con al menos un reinicio: {int((LG.reinicios > 0).sum())} | reinicios totales: {int(LG.reinicios.sum())}")
    print(LG[LG.reinicios > 0][["variante", "semilla", "fold", "lr", "epocas", "reinicios", "perdida_en_revision", "perdida_final"]].to_string(index=False))
colapsadas = []
for v, preds in DISP.items():
    for sem, (pos, p) in preds.items():
        for f in FOLDS:
            m = np.isin(pos, np.where(folds == f)[0])
            if m.any() and f1_macro_idx(Y_IDX[pos][m], p[m]) < 0.3:
                colapsadas.append((v, sem, f, round(f1_macro_idx(Y_IDX[pos][m], p[m]), 3)))
print("Particiones finales con F1 < 0.3 (colapso que no se resolvió):", colapsadas if colapsadas else "ninguna")

## 8. Guardar y descargar los resultados

Los CSV se guardan en tu Drive y en `finetuning_v2_xlmr.zip`, que se descarga. Para llevarlos al repositorio, copia el contenido a `oe4_sintesis/finetuning/resultados/` (sin hacer commit hasta revisarlo). Los nombres no chocan con los de la corrida anterior: la variante se llama `ampliada`, `gtr` o `pooling`.

In [ ]:
DIR_OUT = f"{DIR_DRIVE}/resultados"
os.makedirs(DIR_OUT, exist_ok=True)
if len(RES):
    RES.to_csv(f"{DIR_OUT}/finetuning_v3_{MODELO}_resumen.csv", index=False)
    pd.DataFrame(por_clase, index=CLASES).to_csv(f"{DIR_OUT}/finetuning_v3_{MODELO}_f1_por_intencion.csv")
for v in DISP:
    pd.concat([pd.read_csv(r) for r in sorted(glob.glob(f"{DIR_RUNS}/{MODELO}_{v}_s*_f*.csv"))]).to_csv(f"{DIR_OUT}/predicciones_{MODELO}_{v}.csv", index=False)
sel = [pd.read_csv(r).assign(archivo=os.path.basename(r)) for r in sorted(glob.glob(f"{DIR_RUNS}/seleccion_{MODELO}_ampliada_*.csv"))]
if sel:
    pd.concat(sel).to_csv(f"{DIR_OUT}/seleccion_hiperparametros_{MODELO}_ampliada.csv", index=False)
logs_todos = [pd.read_csv(r) for r in sorted(glob.glob(f"{DIR_RUNS}/log_{MODELO}_*_s*_f*.csv"))]
if logs_todos:
    pd.concat(logs_todos).to_csv(f"{DIR_OUT}/registro_entrenamientos_{MODELO}.csv", index=False)
shutil.make_archive(f"/content/finetuning_v3_{MODELO}", "zip", DIR_OUT)
print("Guardado en:", DIR_OUT, "|", sorted(os.listdir(DIR_OUT)))
try:
    from google.colab import files
    files.download(f"/content/finetuning_v3_{MODELO}.zip")
except Exception as e:
    print("Descarga manual desde Drive:", e)

## Cómo leer el resultado (y qué decir en el informe)

- **`dif_vs_finetuning_anterior`:** F1 de la variante menos el del fine-tuning anterior de este modelo. Es *distinguible de cero* solo si su intervalo **no incluye el 0**. Diferencias menores a unos 0.01 no se interpretan.
- **Variante `ampliada`:** responde si la rejilla corta era el límite del fine-tuning. Si sigue eligiendo el máximo de épocas casi siempre, el techo aún no está establecido.
- **Variantes `gtr` y `pooling`:** se comparan con `ampliada` (misma receta de hiperparámetros, un solo cambio). Usan los hiperparámetros de `ampliada` sin buscar otros; es una decisión declarada y conservadora para la variante que cambia.
- **Frente a n-gramas (0.751):** ubica cuánto falta o sobra respecto a la referencia sin modelo de lenguaje.
- **`sd_entre_semillas`:** si es grande (más de 0.02), el fine-tuning es inestable y el promedio debe leerse con cautela.
- **Qué NO concluye:** que el modelo «entienda» el shiwilu. Compara formas de usar este modelo sobre este corpus. Se probaron varias variantes sin corrección por comparaciones múltiples: leer por dirección, no por centésimas.
- **Esta versión es la estabilizada de la segunda tanda de XLM-R.** Compárala con `v2_ampliada`, `v2_gtr` y `v2_pooling` (columnas `dif_vs_v2_*`): ¿mejora, queda igual o baja? Cualquiera de las tres es un resultado válido.
- **`registro_entrenamientos_xlmr_estable.csv`** guarda la pérdida de cada época de cada entrenamiento. Si hubo reinicios, ahí se ve que la pérdida se quedó cerca de 1.95 (= ln 7, predecir siempre lo mismo), lo que confirma la causa del colapso.
- Los resultados se **reportan junto a** los de la segunda tanda; no los reemplazan.